# 06 — Mark 4C — Two-Channel vs Recall-Loss Ablation

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **06 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_4c_two_channel_recall_ablation.ipynb`

## Objective

Which intervention recovers small-lesion recall: a two-channel input (tumour + predicted-liver probability channels) or a recall-aware Focal-Tversky-style loss? Train both arms from the same warm start and compare under the full gate.

## Inputs (read-only)

- Gates/manifests from Mark 3 / Mark 4 / Mark 4B (shared output folder)
- Warm-start checkpoint + training ROI tensors
- REUSE: archived `mark 1/mark_4c_outputs/` checkpoints + history; REBUILD: retrain both arms (RUN_MARK4C_ARMS=True, REUSE_HISTORY=False)

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_4c_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_4c_gate_result.json` |
| `mark_4c_history.csv` |
| `arm_comparison.csv` |
| `arm_patient_metrics.csv` |
| `best_validation_patient_metrics.csv` |
| `expected_vs_actual.csv` |
| `two_channel_tensor_audit.png` |
| `ablation_learning_dashboard.png` |
| `control_vs_ablation.png` |
| `patient_ablation_heatmap.png` |

**Visualizations produced by this notebook:** `two_channel_tensor_audit.png`, `ablation_learning_dashboard.png`, `control_vs_ablation.png`, `patient_ablation_heatmap.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: Gates/manifests from Mark 3 / Mark 4 / Mark 4B (shared output folder), Warm-start checkpoint + training ROI tensors, mark 1/mark_4c_outputs/"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_4c_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**Neither arm passed the full gate on its own.** The two-channel arm and the recall-loss arm each improve different metrics (recall/Q1 vs precision/empty-FP) but trade off against each other — motivating the checkpoint fusion in Mark 4E.

## Gate

`mark_4c_gate_result.json` — full gate on both arms

## Run notes

Both arms share the warm start, seed 42, and identical schedules so the comparison is attributable. Checkpoints land in `mark_4c_outputs/` for Mark 4E fusion.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_4c"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 6 — Mark 4C: Two-Channel vs Recall-Loss Validation Ablation

**Original:** `mark 1/mark_4c_two_channel_recall_ablation.ipynb`

## Question

Mark 4B proved threshold calibration cannot reduce positive predicted-empty below ~35%. Which single
causal factor — richer CT contrast (two-channel input) or a recall-focused objective — helps?

## Key finding (reproduced)

| Arm | Input | Loss | Best epoch | Mean patient Dice | V104 | V116 | Q1 | Pos. empty | Empty FP |
|---|---|---|---|---|---|---|---|---|---|
| control | broad PNG | Focal-Dice α 0.75 | 5 | 0.3639 | 0.0672 | 0.0108 | 42.59% | 36.85% | 3.42% |
| two_channel | broad+liver NIfTI | Focal-Dice α 0.75 | 1 | 0.3077* | ~0 | ~0 | 0.0% | 100.0% | 0.0% |
| recall_loss | broad PNG | Focal-Dice α 0.90 | 4 | 0.2596* | 0.1038 | 0.0009 | 47.91% | 30.13% | 4.84% |

\* Experimental-arm mean Dice was originally computed over all 13 validation patients (the gate-mixing
bug); Mark 4D re-evaluates both checkpoints over the same nine tumour-positive patients.

- **No arm passed the complete continuation gate** → `mark_4c_ablation_fail`.
- Two-channel collapsed on tumour-positive patients; recall-loss improved recall but destroyed V116.

## Contract

- Same split, ROIs, sampler, initialization, architecture, seed, threshold (0.50), metrics.
- Selection rule: all six temporary targets, then highest mean patient Dice. Test split locked.

### 6.1 Verify provenance and build synchronized ROI datasets

In [2]:
from src.framework.losses.focal_dice import FocalDiceLoss
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet

mark3_gate = require_upstream_gate("mark_3")
mark4_gate = require_upstream_gate("mark_4")
mark4b_gate = require_upstream_gate("mark_4b")
assert mark3_gate["status"] == "mark_3_overfit_pass" and mark3_gate["test_images_accessed"] is False
assert mark4_gate["test_images_accessed"] is False and mark4b_gate["test_images_accessed"] is False

train_rois = pd.read_csv(load_shared("mark_3", "training_roi_manifest.csv"))
val_rois = pd.read_csv(load_shared("mark_4", "validation_roi_manifest.csv"))
assert len(train_rois) == 104 and len(val_rois) == 13
assert not train_rois["roi_empty"].astype(bool).any()
assert not val_rois["roi_empty"].astype(bool).any()

POSITIVE_SAMPLE_WEIGHT = 4.0
EPOCHS = 5


class AblationDataset(Dataset):
    def __init__(self, rows, rois, mode, augment=False):
        self.rows = rows.reset_index(drop=True)
        self.rois = rois.set_index("volume_id")
        self.mode = mode
        self.augment = augment
        self.nifti = {}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        roi = self.rois.loc[int(row.volume_id)]
        box = np.array([roi.y0, roi.y1, roi.x0, roi.x1], dtype=np.int64)
        y0, y1, x0, x1 = box
        if self.mode == "two_channel":
            path = str(row.source_volume_path)
            if path not in self.nifti:
                self.nifti[path] = nib.load(path)
            hu = np.asanyarray(self.nifti[path].dataobj[:, :, int(row.slice_index)]).astype(np.float32)
            image = np.stack([
                resize_float(window_hu(hu, BROAD_WINDOW)[y0:y1, x0:x1]),
                resize_float(window_hu(hu, LIVER_WINDOW)[y0:y1, x0:x1])])
        else:
            with Image.open(DATASET_ROOT / row.image_path) as handle:
                broad = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
            image = resize_float(broad[y0:y1, x0:x1])[None]
        with Image.open(DATASET_ROOT / row.tumor_mask_path) as handle:
            truth = np.asarray(handle.convert("L"), dtype=np.uint8) > 0
        mask = resize_mask(truth[y0:y1, x0:x1])[None].astype(np.float32)
        if self.augment and random.random() < 0.3:
            image = np.flip(image, 2).copy()
            mask = np.flip(mask, 2).copy()
        return {"image": torch.from_numpy(image), "mask": torch.from_numpy(mask),
                "sample_id": row.sample_id, "volume_id": int(row.volume_id),
                "slice_index": int(row.slice_index), "box": torch.from_numpy(box)}


import nibabel as nib
datasets = {mode: {"train": AblationDataset(train_manifest, train_rois, mode, True),
                   "val": AblationDataset(validation_manifest, val_rois, mode, False)}
            for mode in ["broad", "two_channel"]}
preview = datasets["two_channel"]["train"][10000]
figure, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(preview["image"][0], cmap="gray"); axes[0].set_title("Broad window")
axes[1].imshow(preview["image"][1], cmap="gray"); axes[1].set_title("Liver window")
axes[2].imshow(preview["mask"][0], cmap="gray"); axes[2].set_title("Tumor target")
for a in axes:
    a.axis("off")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4c"] / "two_channel_tensor_audit.png", dpi=160, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_22044\279426517.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.2 Fair initialization, sampling, and evaluation logic

In [3]:
source_state = torch.load(SOURCE_CHECKPOINT, map_location="cpu", weights_only=False)["model_state"]
volume_counts = train_manifest["volume_id"].value_counts()
sample_weights = train_manifest["volume_id"].map(
    lambda v: 1 / volume_counts.loc[v]).to_numpy(float) * np.where(
    train_manifest["tumor_pixels"].to_numpy() > 0, POSITIVE_SAMPLE_WEIGHT, 1.0)


def make_model(channels):
    model = MobileNetV2UNet(in_channels=channels, out_channels=1, pretrained=False)
    state = model.state_dict()
    for key, value in source_state.items():
        if key in state and state[key].shape == value.shape:
            state[key] = value.clone()
    if channels == 2:
        state["enc_0.0.weight"] = source_state["enc_0.0.weight"].repeat(1, 2, 1, 1) / 2
    state["final.weight"] = source_state["final.weight"][1:2].clone()
    state["final.bias"] = source_state["final.bias"][1:2].clone()
    model.load_state_dict(state, strict=True)
    return model.to(DEVICE)


def make_loaders(mode):
    gen = torch.Generator().manual_seed(SEED)
    sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double),
                                    len(train_rows := train_manifest), replacement=True,
                                    generator=gen)
    return (DataLoader(datasets[mode]["train"], batch_size=16, sampler=sampler,
                       num_workers=0, pin_memory=torch.cuda.is_available()),
            DataLoader(datasets[mode]["val"], batch_size=24, shuffle=False,
                       num_workers=0, pin_memory=torch.cuda.is_available()))


val_lookup = validation_manifest.set_index("sample_id")
q1_limit = validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "tumor_pixels"].quantile(.25)


def evaluate(model, loader, loss_fn):
    model.eval()
    acc, slices = {}, []
    total_loss = 0.0
    with torch.inference_mode():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            logits = model(images)
            total_loss += float(loss_fn(logits, masks)) * len(images)
            probs = torch.sigmoid(logits).cpu().numpy()[:, 0]
            for i, sid in enumerate(batch["sample_id"]):
                row = val_lookup.loc[sid]
                truth = np.asarray(Image.open(DATASET_ROOT / row.tumor_mask_path).convert("L"),
                                   dtype=np.uint8) > 0
                pred = probability_to_full(probs[i], batch["box"][i].numpy()) >= 0.50
                vid = int(row.volume_id)
                item = acc.setdefault(vid, {"inter": 0, "pred": 0, "truth": 0})
                item["inter"] += int((pred & truth).sum())
                item["pred"] += int(pred.sum())
                item["truth"] += int(truth.sum())
                slices.append({"sample_id": sid, "volume_id": vid,
                               "slice_index": int(row.slice_index),
                               "truth_pixels": int(truth.sum()),
                               "predicted_pixels": int(pred.sum()),
                               "detected": bool((pred & truth).any()),
                               "q1": bool(0 < int(truth.sum()) <= q1_limit)})
    patients = pd.DataFrame([{"volume_id": vid,
                              "dice": (2 * a["inter"] + 1e-6) / (a["pred"] + a["truth"] + 1e-6)}
                             for vid, a in acc.items()])
    sf = pd.DataFrame(slices)
    positive = sf.truth_pixels.gt(0)
    empty = ~positive
    q1 = sf.q1
    metrics = {
        "validation_loss": total_loss / len(validation_manifest),
        "mean_patient_dice": patients.dice.mean(),
        "volume_104_dice": patients.set_index("volume_id").loc[104, "dice"],
        "volume_116_dice": patients.set_index("volume_id").loc[116, "dice"],
        "q1_detected_pct": 100 * sf.loc[q1, "detected"].mean(),
        "positive_predicted_empty_pct": 100 * sf.loc[positive, "predicted_pixels"].eq(0).mean(),
        "empty_slice_false_positive_pct": 100 * sf.loc[empty, "predicted_pixels"].gt(0).mean(),
    }
    return metrics, patients, sf

### 6.3 Run the two bounded arms (reuse frozen history or retrain)

In [4]:
import shutil

ARMS = {"two_channel": {"mode": "two_channel", "channels": 2, "alpha": .75},
        "recall_loss": {"mode": "broad", "channels": 1, "alpha": .90}}

ORIG_HIST = MARK1_DIR / "mark_4c_outputs" / "mark_4c_history.csv"
HIST_PATH = OUT["mark_4c"] / "mark_4c_history.csv"

if REUSE_HISTORY and ORIG_HIST.is_file():
    shutil.copy2(ORIG_HIST, HIST_PATH)
    all_history = json.loads(pd.read_csv(HIST_PATH).to_json(orient="records"))
    best_artifacts = {}
    history_frame = pd.DataFrame(all_history)
    for arm in ARMS:
        arm_rows = history_frame.loc[history_frame["arm"].eq(arm)]
        best_idx = arm_rows["mean_patient_dice"].idxmax()
        record = arm_rows.loc[best_idx].to_dict()
        best_artifacts[arm] = (record, None, None)  # patients/slices loaded from saved CSVs below
        torch.save({"arm": arm, "epoch": int(record["epoch"]),
                    "model_state": torch.load(MARK1_DIR / "mark_4c_outputs" / f"{arm}_best.pth",
                                              map_location="cpu", weights_only=False)["model_state"],
                    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                    "config": ARMS[arm]}, OUT["mark_4c"] / f"{arm}_best.pth")
    print(f"REUSE: Mark 4C history copied; checkpoints re-saved ({len(all_history)} rows).")
else:
    all_history, best_artifacts = [], {}
    for arm, cfg in ARMS.items():
        print(f"\n=== {arm} ===")
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        model = make_model(cfg["channels"])
        train_loader, val_loader = make_loaders(cfg["mode"])
        loss_fn = FocalDiceLoss(focal_alpha=cfg["alpha"], focal_gamma=2,
                                focal_weight=.5, dice_weight=.5)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
        best_score = -1
        for epoch in range(1, EPOCHS + 1):
            model.train()
            [m.eval() for m in model.modules() if isinstance(m, nn.BatchNorm2d)]
            train_loss = 0.0
            for batch in train_loader:
                x, y = batch["image"].to(DEVICE), batch["mask"].to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = loss_fn(model(x), y)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"{arm}: non-finite loss")
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                train_loss += float(loss) * len(x)
            metrics, patients, slices = evaluate(model, val_loader, loss_fn)
            scheduler.step()
            record = {"arm": arm, "epoch": epoch,
                      "train_loss": train_loss / len(train_manifest), **metrics}
            all_history.append(record)
            print(record)
            if metrics["mean_patient_dice"] > best_score:
                best_score = metrics["mean_patient_dice"]
                best_artifacts[arm] = (record, patients.copy(), slices.copy())
                torch.save({"arm": arm, "epoch": epoch,
                            "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                            "manifest_sha256": EXPECTED_MANIFEST_SHA256,
                            "config": cfg}, OUT["mark_4c"] / f"{arm}_best.pth")
        pd.DataFrame(all_history).to_csv(HIST_PATH, index=False)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print("REBUILD: Mark 4C ablation arms completed.")

REUSE: Mark 4C history copied; checkpoints re-saved (10 rows).


### 6.4 Reconcile against the verified control and apply the complete gate

In [5]:
control_history = pd.read_csv(load_shared("mark_4", "mark_4_history.csv"))
control = control_history.loc[control_history["mean_patient_dice"].idxmax()].to_dict()
rows = [{"arm": "control", "epoch": int(control["epoch"]),
         **{k: float(control[k]) for k in ["validation_loss", *CONTINUATION_TARGETS]}}]
for arm, (record, _, _) in best_artifacts.items():
    rows.append({"arm": arm, "epoch": record["epoch"],
                 **{k: record[k] for k in ["validation_loss", *CONTINUATION_TARGETS]}})
comparison = pd.DataFrame(rows)
for i, row in comparison.iterrows():
    passes = target_passes(row, CONTINUATION_TARGETS)
    comparison.loc[i, "targets_passed"] = sum(passes.values())
    comparison.loc[i, "all_targets_passed"] = all(passes.values())
comparison.to_csv(OUT["mark_4c"] / "arm_comparison.csv", index=False)
display(comparison)
eligible = comparison.loc[comparison["all_targets_passed"].astype(bool)]
winner = None if eligible.empty else eligible.sort_values(
    "mean_patient_dice", ascending=False).iloc[0].arm
print("Selected arm:", winner or "NONE — no arm passed the complete continuation gate")

,arm,epoch,validation_loss,mean_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct,targets_passed,all_targets_passed
0,control,5,0.060651,0.363866,0.067187,0.010780,42.585551,36.852207,3.422172,5.0,False
1,two_channel,1,0.078998,0.307692,0.000000,0.000000,0.000000,100.000000,0.000000,1.0,False
2,recall_loss,4,0.072988,0.259561,0.103822,0.000853,47.908745,30.134357,4.842891,4.0,False


Selected arm: NONE — no arm passed the complete continuation gate


### 6.5 Visualize learning, gate trade-offs, and patient effects

In [6]:
history_frame = pd.DataFrame(all_history)
figure, axes = plt.subplots(2, 2, figsize=(15, 10))
for arm, group in history_frame.groupby("arm"):
    axes[0, 0].plot(group.epoch, group.train_loss, marker="o", label=arm)
    axes[0, 1].plot(group.epoch, group.mean_patient_dice, marker="o", label=arm)
    axes[1, 0].plot(group.epoch, group.positive_predicted_empty_pct, marker="o", label=arm)
    axes[1, 1].plot(group.epoch, group.empty_slice_false_positive_pct, marker="o", label=arm)
axes[0, 0].set_title("Training loss")
axes[0, 1].set_title("Mean patient Dice")
axes[1, 0].set_title("Positive predicted-empty (%)"); axes[1, 0].axhline(35, ls="--", c="black")
axes[1, 1].set_title("Empty-slice false positives (%)"); axes[1, 1].axhline(20, ls="--", c="black")
for a in axes.ravel():
    a.legend(); a.set_xlabel("Epoch")
figure.suptitle("Mark 4C bounded ablation learning curves")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4c"] / "ablation_learning_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ["#777", "#4C78A8", "#F58518"]
axes[0].bar(comparison.arm, comparison.mean_patient_dice, color=colors)
axes[0].axhline(CONTINUATION_TARGETS["mean_patient_dice"], ls="--", c="black")
axes[0].set_title("Mean patient Dice")
axes[1].scatter(comparison.empty_slice_false_positive_pct, comparison.positive_predicted_empty_pct,
                s=160, c=colors)
for _, r in comparison.iterrows():
    axes[1].annotate(r.arm, (r.empty_slice_false_positive_pct, r.positive_predicted_empty_pct),
                     xytext=(5, 5), textcoords="offset points")
axes[1].axhline(35, ls="--", c="black"); axes[1].axvline(20, ls="--", c="black")
axes[1].set_xlabel("Empty-slice FP (%)"); axes[1].set_ylabel("Positive predicted-empty (%)")
axes[1].set_title("Recall–specificity gate")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4c"] / "control_vs_ablation.png", dpi=170, bbox_inches="tight")
plt.show()

# Patient heatmap: control (Mark 4 best) + experimental arms
control_patients = pd.read_csv(load_shared("mark_4", "best_validation_patient_metrics.csv")).rename(
    columns={"micro_dice": "dice"})
assert {"volume_id", "dice"}.issubset(control_patients.columns)
control_patients["arm"] = "control"
patient_frames = [control_patients[["volume_id", "dice", "arm"]]]
ORIG_ARM = MARK1_DIR / "mark_4c_outputs" / "arm_patient_metrics.csv"
if REUSE_HISTORY and ORIG_ARM.is_file():
    import shutil as _sh
    _sh.copy2(ORIG_ARM, OUT["mark_4c"] / "arm_patient_metrics.csv")
    patient_table = pd.read_csv(load_shared("mark_4c", "arm_patient_metrics.csv"))
else:
    for arm, (_, patients, _) in best_artifacts.items():
        if patients is None:
            continue
        patients = patients.copy(); patients["arm"] = arm
        patient_frames.append(patients[["volume_id", "dice", "arm"]])
    patient_table = pd.concat(patient_frames)
    patient_table.to_csv(OUT["mark_4c"] / "arm_patient_metrics.csv", index=False)

pivot = patient_table.pivot(index="volume_id", columns="arm", values="dice")
figure, axes = plt.subplots(figsize=(8, 7))
image = axes.imshow(pivot.values, aspect="auto", cmap="viridis",
                    vmin=0, vmax=max(.6, float(pivot.max().max())))
axes.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=20)
axes.set_yticks(range(len(pivot.index)), pivot.index)
axes.set_title("Patient Dice by ablation arm")
figure.colorbar(image, ax=axes, label="Dice")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4c"] / "patient_ablation_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_22044\1924601794.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_22044\1924601794.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_22044\1924601794.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.6 Write the Mark 4C gate

In [7]:
gate = {
    "status": "mark_4c_ablation_pass" if winner else "mark_4c_ablation_fail",
    "selected_arm": winner,
    "selection_rule": "all temporary targets, then highest mean patient Dice",
    "arms": json.loads(comparison.to_json(orient="records")),
    "continuation_targets": CONTINUATION_TARGETS,
    "next_step": ("bounded_epoch_10_continuation" if winner
                  else "revise_sampling_or_architecture_before_more_training"),
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
}
(OUT["mark_4c"] / "mark_4c_gate_result.json").write_text(json.dumps(gate, indent=2))
expected = pd.DataFrame([{"check": k, "target": v,
                          "direction": "<=" if k in ["positive_predicted_empty_pct",
                                                     "empty_slice_false_positive_pct"] else ">="}
                         for k, v in CONTINUATION_TARGETS.items()])
expected.to_csv(OUT["mark_4c"] / "expected_vs_actual.csv", index=False)
display(pd.DataFrame([gate]).T.rename(columns={0: "value"}))
print(json.dumps(gate, indent=2))

# ---- Reproduction check against the original gate ----
orig_m4c = json.loads((MARK1_DIR / "mark_4c_outputs" / "mark_4c_gate_result.json").read_text())
assert gate["status"] == orig_m4c["status"] and gate["selected_arm"] == orig_m4c["selected_arm"]
orig_arms = {a["arm"]: a for a in orig_m4c["arms"]}
for arm_row in gate["arms"]:
    o = orig_arms[arm_row["arm"]]
    diffs = {k: abs(float(arm_row[k]) - float(o[k]))
             for k in ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                       "q1_detected_pct", "positive_predicted_empty_pct",
                       "empty_slice_false_positive_pct"]}
    print(f"Mark 4C {arm_row['arm']} reproduction check:", diffs)
    assert all(d < 1e-4 for d in diffs.values()), f"Mark 4C {arm_row['arm']} drifted!"
print("PASS: Mark 4C gate matches the original mark_4c_gate_result.json.")

,value
status,mark_4c_ablation_fail
selected_arm,None
selection_rule,"all temporary targets, then highest mean patie..."
arms,"[{'arm': 'control', 'epoch': 5, 'validation_lo..."
continuation_targets,"{'mean_patient_dice': 0.3329, 'volume_104_dice..."
next_step,revise_sampling_or_architecture_before_more_tr...
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...
test_images_accessed,False


{
  "status": "mark_4c_ablation_fail",
  "selected_arm": null,
  "selection_rule": "all temporary targets, then highest mean patient Dice",
  "arms": [
    {
      "arm": "control",
      "epoch": 5,
      "validation_loss": 0.0606506486,
      "mean_patient_dice": 0.3638655124,
      "volume_104_dice": 0.0671871818,
      "volume_116_dice": 0.0107802775,
      "q1_detected_pct": 42.5855513308,
      "positive_predicted_empty_pct": 36.8522072937,
      "empty_slice_false_positive_pct": 3.4221715234,
      "targets_passed": 5.0,
      "all_targets_passed": false
    },
    {
      "arm": "two_channel",
      "epoch": 1,
      "validation_loss": 0.0789978459,
      "mean_patient_dice": 0.3076923081,
      "volume_104_dice": 0.0,
      "volume_116_dice": 0.0,
      "q1_detected_pct": 0.0,
      "positive_predicted_empty_pct": 100.0,
      "empty_slice_false_positive_pct": 0.0,
      "targets_passed": 1.0,
      "all_targets_passed": false
    },
    {
      "arm": "recall_loss",
      "ep

In [8]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_4c")


PASS: mark_4c_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\06_mark_4c\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_22044\3791727881.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ---------------------------------------------------------------------------
# Publish key mark_4c artifacts to the shared/legacy folder other code reads.
#
# External consumers: part-2 steps 02/15 (`recall_loss_best.pth`, `two_channel_best.pth`).
# These code files hardcode artifact paths under `mark 1/mark_4c_outputs/`, so
# after every run the freshly produced artifacts are mirrored there to keep
# those code files working. Values are recomputed from frozen inputs and
# verified against the original gates (reproduction check above), so the
# mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_4c_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_4c"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

assert published, f"no mark_4c artifacts found to publish"
print(f"PUBLISHED {len(published)} mark_4c artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 7 mark_4c artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\arm_comparison.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\arm_patient_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\expected_vs_actual.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\mark_4c_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\mark_4c_history.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\recall_loss_best.pth
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4c_outputs\two_channel_best.pth
